# FlyWire skeleton 3D visualization notebook

This notebook builds an interactive 3D visualization from:

- `cluster_assignment_dict.json`
- a folder of neuron skeleton `.swc` files

It supports:

- gray background skeleton layer
- single-cluster highlight
- multi-cluster highlight
- Plotly interaction in notebook
- HTML export

## 1. Install dependencies

Run this only if needed.

## 2. Imports

In [1]:
import os
import json
import random
from pathlib import Path
from collections import defaultdict

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
import ipywidgets as widgets

## 3. Config

Change these paths to your own.

In [2]:
# ===== paths =====
cluster_json_path = Path("runs_wsbm/sbm_0/cluster_assignment_dict.json")
swc_folder = Path("swc_folder")   # folder containing many .swc files

# ===== visualization settings =====
background_sample_neurons = 500
max_points_per_neuron_bg = 80
max_points_per_neuron_fg = 400
default_foreground_neurons = 80
random_seed = 42


# background_sample_neurons = 30
# max_points_per_neuron_bg = 10
# max_points_per_neuron_fg = 40
# default_foreground_neurons = 5
# random_seed = 42


random.seed(random_seed)

print("cluster_json_path =", cluster_json_path)
print("swc_folder         =", swc_folder)

cluster_json_path = runs_wsbm\sbm_0\cluster_assignment_dict.json
swc_folder         = swc_folder


## 4. Load clustering result

In [3]:
with open(cluster_json_path, "r") as f:
    cluster_assignment = json.load(f)

cluster_assignment = {int(k): int(v) for k, v in cluster_assignment.items()}

print("Loaded neurons in clustering result:", len(cluster_assignment))

cluster_to_rootids = defaultdict(list)
for rid, cid in cluster_assignment.items():
    cluster_to_rootids[cid].append(rid)

cluster_sizes = sorted(
    [(cid, len(rids)) for cid, rids in cluster_to_rootids.items()],
    key=lambda x: (-x[1], x[0])
)

print("Number of clusters:", len(cluster_sizes))
print("Top 10 largest clusters:")
cluster_sizes[:10]

Loaded neurons in clustering result: 138639
Number of clusters: 1085
Top 10 largest clusters:


[(795, 2268),
 (512, 2043),
 (619, 1808),
 (745, 1413),
 (834, 699),
 (560, 655),
 (429, 620),
 (739, 589),
 (840, 580),
 (987, 546)]

## 5. SWC reader

Typical SWC format:

`node_id type x y z radius parent_id`

In [4]:
def read_swc(filepath):
    rows = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            parts = line.split()
            if len(parts) < 7:
                continue

            node_id = int(float(parts[0]))
            x = float(parts[2])
            y = float(parts[3])
            z = float(parts[4])
            parent_id = int(float(parts[6]))

            rows.append({
                "node_id": node_id,
                "parent_id": parent_id,
                "x": x,
                "y": y,
                "z": z
            })

    return pd.DataFrame(rows)

## 6. Convert one SWC neuron into line coordinates for Plotly

In [5]:
def swc_to_line_xyz(df, max_points=None, random_state=42):
    if df.empty:
        return [], [], []

    if max_points is not None and len(df) > max_points:
        keep_n = max(1, max_points)
        sampled_df = df.sample(n=keep_n, random_state=random_state).copy()

        root_rows = df[df["parent_id"] == -1]
        if not root_rows.empty:
            sampled_df = pd.concat([sampled_df, root_rows], axis=0).drop_duplicates(subset=["node_id"])

        df = sampled_df.copy()

    node_map = {
        int(row["node_id"]): (row["x"], row["y"], row["z"])
        for _, row in df.iterrows()
    }

    x_lines, y_lines, z_lines = [], [], []

    for _, row in df.iterrows():
        parent_id = int(row["parent_id"])
        node_id = int(row["node_id"])

        if parent_id == -1:
            continue
        if parent_id not in node_map or node_id not in node_map:
            continue

        x1, y1, z1 = node_map[node_id]
        x2, y2, z2 = node_map[parent_id]

        x_lines.extend([x1, x2, None])
        y_lines.extend([y1, y2, None])
        z_lines.extend([z1, z2, None])

    return x_lines, y_lines, z_lines

## 7. Helper to load one neuron

In [6]:
# def load_neuron_swc(root_id, swc_folder):
#     filepath = swc_folder / f"{root_id}.swc"
#     if not filepath.exists():
#         return None
#     df = read_swc(filepath)
#     if df.empty:
#         return None
#     return df

In [7]:
swc_cache = {}
def load_neuron_swc(root_id, swc_folder):
    if root_id in swc_cache:
        return swc_cache[root_id]

    filepath = swc_folder / f"{root_id}.swc"
    if not filepath.exists():
        return None

    df = read_swc(filepath)
    if df.empty:
        return None

    swc_cache[root_id] = df
    return df

## 8. Check which clustering neurons actually have SWC files

In [8]:
available_rootids = []
missing_rootids = []

for rid in cluster_assignment.keys():
    if (swc_folder / f"{rid}.swc").exists():
        available_rootids.append(rid)
    else:
        missing_rootids.append(rid)

print("SWC available:", len(available_rootids))
print("SWC missing:", len(missing_rootids))
print("Missing ratio:", round(len(missing_rootids) / max(1, len(cluster_assignment)), 4))

SWC available: 138639
SWC missing: 0
Missing ratio: 0.0


## 9. Sample neurons for background layer

In [9]:
bg_rootids = random.sample(
    available_rootids,
    min(background_sample_neurons, len(available_rootids))
)

print("Background neurons sampled:", len(bg_rootids))
bg_rootids[:10]

Background neurons sampled: 500


[720575940630546171,
 720575940621961315,
 720575940619986486,
 720575940622213953,
 720575940614620831,
 720575940627417689,
 720575940619029680,
 720575940628452201,
 720575940608295726,
 720575940622078049]

## 10. Build gray background trace

In [10]:
def build_background_trace(bg_rootids, swc_folder, max_points_per_neuron=80):
    x_all, y_all, z_all = [], [], []

    for rid in bg_rootids:
        df = load_neuron_swc(rid, swc_folder)
        if df is None:
            continue

        x, y, z = swc_to_line_xyz(df, max_points=max_points_per_neuron, random_state=42)
        x_all.extend(x)
        y_all.extend(y)
        z_all.extend(z)

    trace = go.Scatter3d(
        x=x_all,
        y=y_all,
        z=z_all,
        mode="lines",
        line=dict(color="rgba(150,150,150,0.18)", width=1),
        name="background",
        hoverinfo="skip"
    )
    return trace

background_trace = build_background_trace(
    bg_rootids,
    swc_folder,
    max_points_per_neuron=max_points_per_neuron_bg
)

print("Background trace ready.")

Background trace ready.


## 11. Build one cluster trace

In [11]:
def build_cluster_trace(
    cluster_id,
    cluster_to_rootids,
    swc_folder,
    color="red",
    max_neurons=None,
    max_points_per_neuron=400,
    random_state=42
):
    rootids = list(cluster_to_rootids[cluster_id])
    rootids = [rid for rid in rootids if (swc_folder / f"{rid}.swc").exists()]

    if max_neurons is not None and len(rootids) > max_neurons:
        rng = random.Random(random_state)
        rootids = rng.sample(rootids, max_neurons)

    x_all, y_all, z_all = [], [], []

    for rid in rootids:
        df = load_neuron_swc(rid, swc_folder)
        if df is None:
            continue

        x, y, z = swc_to_line_xyz(df, max_points=max_points_per_neuron, random_state=random_state)
        x_all.extend(x)
        y_all.extend(y)
        z_all.extend(z)

    trace = go.Scatter3d(
        x=x_all,
        y=y_all,
        z=z_all,
        mode="lines",
        line=dict(color=color, width=3),
        name=f"cluster {cluster_id}",
        hoverinfo="name"
    )
    return trace

## 12. Preview one cluster

In [12]:
test_cluster = cluster_sizes[0][0]

test_trace = build_cluster_trace(
    test_cluster,
    cluster_to_rootids,
    swc_folder,
    color="red",
    max_neurons=default_foreground_neurons,
    max_points_per_neuron=max_points_per_neuron_fg
)

fig = go.Figure(data=[background_trace, test_trace])

fig.update_layout(
    width=1000,
    height=800,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    title=f"Cluster {test_cluster}"
)

fig.show()

## 13. Interactive single-cluster viewer

In [13]:
cluster_dropdown = widgets.Dropdown(
    options=[cid for cid, _ in cluster_sizes],
    value=cluster_sizes[0][0],
    description="Cluster:"
)

max_neurons_slider = widgets.IntSlider(
    value=default_foreground_neurons,
    min=10,
    max=300,
    step=10,
    description="Neurons:"
)

color_picker = widgets.ColorPicker(
    value="#ff0000",
    description="Color:"
)

background_toggle = widgets.Checkbox(
    value=True,
    description="Show background"
)

out_single = widgets.Output()

In [14]:
def render_cluster(cluster_id, max_neurons, color, show_background=True):
    with out_single:
        out_single.clear_output(wait=True)

        fg_trace = build_cluster_trace(
            cluster_id,
            cluster_to_rootids,
            swc_folder,
            color=color,
            max_neurons=max_neurons,
            max_points_per_neuron=max_points_per_neuron_fg
        )

        traces = [fg_trace]
        if show_background:
            traces = [background_trace, fg_trace]

        fig = go.Figure(data=traces)

        fig.update_layout(
            width=1000,
            height=800,
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data"
            ),
            margin=dict(l=0, r=0, b=0, t=40),
            title=f"Cluster {cluster_id}"
        )
        fig.show()

In [15]:
ui_single = widgets.VBox([
    cluster_dropdown,
    max_neurons_slider,
    color_picker,
    background_toggle
])

def on_single_change(change=None):
    render_cluster(
        cluster_dropdown.value,
        max_neurons_slider.value,
        color_picker.value,
        background_toggle.value
    )

cluster_dropdown.observe(on_single_change, names="value")
max_neurons_slider.observe(on_single_change, names="value")
color_picker.observe(on_single_change, names="value")
background_toggle.observe(on_single_change, names="value")

display(ui_single, out_single)
on_single_change()

Output()

## 14. Interactive multi-cluster viewer

In [16]:
# # multi_select = widgets.SelectMultiple(
# #     options=[cid for cid, _ in cluster_sizes[:50]],
# #     value=(cluster_sizes[0][0],),
# #     description="Clusters",
# #     rows=12
# # )


# multi_select = widgets.SelectMultiple(
#     options=[cid for cid, _ in cluster_sizes],
#     value=(cluster_sizes[0][0],),
#     description="Clusters",
#     rows=12
# )


# multi_max_neurons_slider = widgets.IntSlider(
#     value=80,
#     min=10,
#     max=200,
#     step=10,
#     description="Per-cluster"
# )

# multi_background_toggle = widgets.Checkbox(
#     value=True,
#     description="Show background"
# )

# out_multi = widgets.Output()

# palette = [
#     "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
#     "#ffff33", "#a65628", "#f781bf", "#999999", "#1b9e77",
#     "#d95f02", "#7570b3"
# ]

In [17]:
cluster_search = widgets.Text(
    value="",
    placeholder="Type cluster id...",
    description="Search:",
    layout=widgets.Layout(width="300px")
)

multi_select = widgets.SelectMultiple(
    options=[cid for cid, _ in cluster_sizes],
    value=(cluster_sizes[0][0],),
    description="Clusters",
    rows=12,
    layout=widgets.Layout(width="300px", height="260px")
)

multi_max_neurons_slider = widgets.IntSlider(
    value=80,
    min=10,
    max=200,
    step=10,
    description="Per-cluster"
)

multi_background_toggle = widgets.Checkbox(
    value=True,
    description="Show background"
)

out_multi = widgets.Output()

palette = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
    "#ffff33", "#a65628", "#f781bf", "#999999", "#1b9e77",
    "#d95f02", "#7570b3"
]

all_cluster_ids = [cid for cid, _ in cluster_sizes]

In [18]:
def update_cluster_options(change=None):
    query = cluster_search.value.strip()

    if query == "":
        filtered = all_cluster_ids[:200]
    else:
        filtered = [cid for cid in all_cluster_ids if query in str(cid)]

    old_selected = list(multi_select.value)
    new_selected = tuple([cid for cid in old_selected if cid in filtered])

    multi_select.options = filtered
    multi_select.value = new_selected

In [19]:
def render_multiple_clusters(selected_clusters, per_cluster_neurons=80, show_background=True):
    with out_multi:
        out_multi.clear_output(wait=True)

        traces = []
        if show_background:
            traces.append(background_trace)

        for i, cid in enumerate(selected_clusters):
            trace = build_cluster_trace(
                cid,
                cluster_to_rootids,
                swc_folder,
                color=palette[i % len(palette)],
                max_neurons=per_cluster_neurons,
                max_points_per_neuron=max_points_per_neuron_fg
            )
            traces.append(trace)

        fig = go.Figure(data=traces)

        fig.update_layout(
            width=1100,
            height=850,
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data"
            ),
            margin=dict(l=0, r=0, b=0, t=40),
            title="Selected clusters"
        )
        fig.show()

In [20]:
ui_multi = widgets.VBox([
    cluster_search,
    multi_select,
    multi_max_neurons_slider,
    multi_background_toggle
])

def on_multi_change(change=None):
    render_multiple_clusters(
        list(multi_select.value),
        per_cluster_neurons=multi_max_neurons_slider.value,
        show_background=multi_background_toggle.value
    )

cluster_search.observe(update_cluster_options, names="value")
multi_select.observe(on_multi_change, names="value")
multi_max_neurons_slider.observe(on_multi_change, names="value")
multi_background_toggle.observe(on_multi_change, names="value")

update_cluster_options()
display(ui_multi, out_multi)
on_multi_change()

Output()

In [21]:
# ui_multi = widgets.VBox([
#     multi_select,
#     multi_max_neurons_slider,
#     multi_background_toggle
# ])

# def on_multi_change(change=None):
#     render_multiple_clusters(
#         list(multi_select.value),
#         per_cluster_neurons=multi_max_neurons_slider.value,
#         show_background=multi_background_toggle.value
#     )

# multi_select.observe(on_multi_change, names="value")
# multi_max_neurons_slider.observe(on_multi_change, names="value")
# multi_background_toggle.observe(on_multi_change, names="value")

# display(ui_multi, out_multi)
# on_multi_change()

## 15. Export one figure to HTML

In [22]:
# Example: export the last manually created figure
fig.write_html("flywire_cluster_view.html")

## 16. save a fixed view for a chosen cluster

In [26]:
# export_cluster_id = cluster_sizes[0][0]
export_cluster_id = 61
export_max_neurons = 100
export_color = "red"

export_trace = build_cluster_trace(
    export_cluster_id,
    cluster_to_rootids,
    swc_folder,
    color=export_color,
    max_neurons=export_max_neurons,
    max_points_per_neuron=max_points_per_neuron_fg
)

export_fig = go.Figure(data=[background_trace, export_trace])
export_fig.update_layout(
    width=1000,
    height=800,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    title=f"Cluster {export_cluster_id}"
)

export_fig.show()

# Uncomment to save:
export_fig.write_html(f"cluster_{export_cluster_id}_view.html")

In [24]:
swc_cache.clear()
print("cache cleared")

cache cleared


## 17. Notes

- If rendering is slow, lower:
  - `background_sample_neurons`
  - `max_points_per_neuron_bg`
  - `max_points_per_neuron_fg`
  - the foreground neuron count slider
- If some neurons do not appear, check whether their `.swc` files exist.
- This version uses only skeletons, so the background is also a skeleton background, not a translucent brain mesh.
- Later, you can add a gray brain mesh layer on top of this pipeline.